## Lab - Using Interpretable Features for Model Debugging

In our lecture on interpretable features, you learned about the importance of using interpretable features for interpretable models. As we talked about, one of the benefits of interpretable machien learning is it makes it easier to identify the source of a problem with a deployed model.

In this lab, we'll be working with a modified version of the [Ames Housing Dataset](https://www.kaggle.com/c/house-prices-advanced-regression-techniques). In this dataset, each row represents a single house in Ames Iowa, and our model is traiend to predict the final sale price of the house. 

In our hypothetical scenario, a real estate agent has deployed a model to help they set the initial sale price houses newly on the market. However, despite decent train and test scores during model training, the agent has found that the house predictions are terrible in the real world. They suspect something went wrong with the data, and want to use methods from *XAI* (explainable AI) to identify the problem.

Your task will be to try to find the issue with the data. You will try transforming the data in different ways to train multiple models with different levels of *feature interpretability*. There are **two mistakes** intentionally added to the dataset. 

### Data Setup

Run the cells below to install libraries, load in the (flawed) dataset, and take an initial look at the data

In [1]:
%pip install lightgbm
%pip install featuretools
%pip install pyreal==0.3.1.1
%pip install matplotlib==3.5.3
%pip install numpy==1.23.5
%pip install shap

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
     ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
     ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
     ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
     ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
     ------- ---

ERROR: Ignored the following versions that require a different python version: 0.2.0 Requires-Python >=3.7.1,<3.10; 0.3.0 Requires-Python >=3.8,<3.11; 0.3.1 Requires-Python >=3.8,<3.11; 0.3.1.1 Requires-Python >=3.8,<3.11; 0.3.2 Requires-Python >=3.8,<3.11; 0.4.0 Requires-Python >=3.8,<3.11; 0.4.1 Requires-Python >=3.8,<3.11; 0.4.10 Requires-Python >=3.9,<3.12; 0.4.2 Requires-Python >=3.8,<3.11; 0.4.3 Requires-Python >=3.8,<3.11; 0.4.4 Requires-Python >=3.8,<3.11; 0.4.5 Requires-Python >=3.9,<3.12; 0.4.6 Requires-Python >=3.9,<3.12; 0.4.7 Requires-Python >=3.9,<3.12; 0.4.8 Requires-Python >=3.9,<3.12; 0.4.9 Requires-Python >=3.9,<3.12
ERROR: Could not find a version that satisfies the requirement pyreal==0.3.1.1 (from versions: 0.1.0.dev0, 0.1.0)
ERROR: No matching distribution found for pyreal==0.3.1.1


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/35.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/35.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/35.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/35.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/35.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/35.2 MB ? eta -:--:--
     ---------------------------------------- 0.3/35.2 MB ? eta -:--:--
     ---------------------------------------- 0.3/35.2 MB ? eta -:--:--
      -------------------------------------- 0.5/35.2 MB 524.3 kB/s eta 0:01:07
      -------------------------------------- 0.5/35.2 MB 524.3 kB/s eta 0:01:07
      -------------------------------------- 0.8/35.2 MB 569.3 kB/s eta 0:01:01
      -------------------------------------- 0.8/35.2 MB 569.3 kB/s eta 0:01:01
     - ------------------------------------

  error: subprocess-exited-with-error
  
  exit code: 1
  
  [585 lines of output]
  <string>:70: SetuptoolsDeprecationWarning: The test command is disabled and references to it are deprecated.
  !!
  
          ********************************************************************************
          Please remove any references to `setuptools.command.test` in all supported versions of the affected package.
  
          This deprecation is overdue, please update your project and remove deprecated
          calls to avoid build errors in the future.
          ********************************************************************************
  
  !!
  C:\Users\Lenovo\AppData\Local\Temp\pip-build-env-bfk13yk9\overlay\Lib\site-packages\setuptools\dist.py:810: SetuptoolsDeprecationWarning: The namespace_packages parameter is deprecated.
  !!
  
          ********************************************************************************
          Please replace its usage with implicit namespace

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
     ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
     ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
      --------------------------------------- 0.3/10.7 MB ? eta -:--:--
      --------------------------------------- 0.3/10.7 MB ? eta -:--:--
      --------------------------------------- 0.3/10.7 MB ? eta -:--:--
     - ------------------------------------- 0.5/10.7 MB 466.4 kB/s eta 0:00:22
     -- ------------------------------------ 0.8/10.7 MB 645.7 kB/s eta 0:00:16
     -- ------------------------------------ 0.8/10.7 MB 645.7 kB/s eta 0:00:16
     --- ----------------------------------- 1.0/10.7 MB 599.4 kB/s eta 0:00:17
     --- ----------------------------------- 1.0/10.7 MB 599.4 kB/s eta 0:00:17
     --- ----------------------------------- 1.0/10.7 MB 599.4 kB/s eta 0:00:17
     ---- -----------------

  error: subprocess-exited-with-error
  
  Getting requirements to build wheel did not run successfully.
  exit code: 1
  
  [33 lines of output]
  Traceback (most recent call last):
    File "E:\anaconda\envs\dcai\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
      main()
    File "E:\anaconda\envs\dcai\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
      json_out["return_val"] = hook(**hook_input["kwargs"])
                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    File "E:\anaconda\envs\dcai\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 137, in get_requires_for_build_wheel
      backend = _build_backend()
                ^^^^^^^^^^^^^^^^
    File "E:\anaconda\envs\dcai\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 70, in _build_backend
      obj = import_module(mod_path)
            ^^^^^^^^^^^^^^^^^^^^^^^
    File 

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
     --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
     --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
     --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
     --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
     ------- -------------------------------- 0.5/2.8 MB 305.2 kB/s eta 0:00:08
     --

In [3]:
import pandas as pd
import numpy as np
import lightgbm
from pyreal.transformers import OneHotEncoder, DataFrameWrapper, fit_transformers, run_transformers
import matplotlib.pyplot as plt

def one_hot_encode(X_train, X_test):
    categorical_features = X_train.select_dtypes(exclude=[np.number]).columns
    ohe = OneHotEncoder(columns=categorical_features)
    
    ohe.fit(pd.concat((X_train, X_test), axis=0))

    return ohe.transform(X_train), ohe.transform(X_test)

def fit_model(X_train, y_train):
    return lightgbm.LGBMRegressor().fit(X_train, y_train)

train_data = pd.read_csv("train_data.csv")
test_data = pd.read_csv("test_data.csv")

y_train = train_data.SalePrice
X_train = train_data.drop("SalePrice", axis=1)

y_test = test_data.SalePrice
X_test = test_data.drop("SalePrice", axis=1)

X_train_model, X_test_model = one_hot_encode(X_train, X_test)
feature_df = pd.read_csv("feature_descriptions.csv") # includes information about features

X_train.head()

ModuleNotFoundError: No module named 'pyreal.transformers'

### Generating Explanations

To generate explanations throughout this lab, we will be using the popular explanation library [shap](https://shap.readthedocs.io/en/latest/). SHAP is a *model agnostic* explanation tool, meaning it can be used to explain any ML model, regardless of architecture --- however, as we will see, its explanations are less than useful when using uninterpretable features.

The shap library offers many different (plotting options)[https://shap.readthedocs.io/en/latest/api_examples.html#plots], you can try multiple out to see what works best for your task. Some plots provide *local explantions*, which explain the logic behind a single prediction. For this lab, you will be working with *global explanations*, which explain the model's logic overall. 

As a start, we recommend you try the **beeswarm** plot. This chart lists the feature the model considers most important on top, with reducing importance going down the y-axis. Each point represents a line in the dataset, with the color representing the value of the feature. The SHAP impact value, on the x-axis represents how much that feature value contributed to the model prediction. A higher value along the x-axis means that feature was highly positively correlated with the model prediction, while a lower value means it was negatively correlated. A SHAP value of zero means that feature value had no effect on the model prediction.

For example, in the sample visualization below, higher median income and lower populations in house blocks tends to result in higher house prices (which makes sense!) 

![example explanation](example_explanation.png)

Run the cell below for two helper functions that will get you started on generating explanations.

In [ ]:
import shap

def explain(model, X_train):
    """
    Explain a model with a beeswarm plot. 
    
    model: a machine learning model
    X_train: a dataset of valid inputs to model
    """
    explainer = shap.explainers.Tree(model, X_train)
    shap_values = explainer(X_train)

    shap.plots.beeswarm(shap_values, max_display=10)

def explain_feature(model, X_train, feature_name):
    """
    Generate an explanation for the impact of a specific 
    feature on model predictions.
    
    model: a machine learning model
    X_train: a dataset of valid inputs to model
    feature_name: string, the name of a feature to investigate
    """
    explainer = shap.explainers.Tree(model, X_train)
    shap_values = explainer(X_train)

    shap.plots.scatter(shap_values[:, feature_name])

### Feature Transforms 1: PCA

We will begin by running a [PCA (Principle Component Analysis)](https://builtin.com/data-science/step-step-explanation-principal-component-analysis) on the data. Run the cell below to standardize and PCA transform the data, and train a light GBM model. Note the performance (given as an [R^2](https://www.geeksforgeeks.org/ml-r-squared-in-regression-analysis/) score).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = DataFrameWrapper(StandardScaler())

X_train_std = scaler.fit_transform(X_train_model)
X_test_std = scaler.transform(X_test_model)

pca = PCA(n_components=0.95)
pca.fit(X_train_std)

X_train_pca = pd.DataFrame(pca.transform(X_train_std), columns=pca.get_feature_names_out())
X_test_pca = pd.DataFrame(pca.transform(X_test_std), columns=pca.get_feature_names_out())

model_pca = fit_model(X_train_pca, y_train)

print("Number of PCA components used: %i" % pca.n_components_)
print("R^2 Score (PCA): %.3f" % model_pca.score(X_test_pca, y_test))

In the cell below, use the explanation helper functions (or others from the shap library/other explanation libraries you may be interested in trying) to generate some explanations of the model's prediction. Think about what additional steps you may need to take to identify the source of poor performance. 

In [ ]:
# Generate an explanation of the PCA model, and see if you can identify the data flaw

explain(model_pca, X_train_pca)
explain_feature(model_pca, X_train_pca, "pca0")

### Feature Transforms 2: Automatic Feature Engineering

Next, we will try a different set of transformations using an automatic feature engineering tool - [Featuretools](https://www.featuretools.com/). This is a powerful library that makes it possible to dry hundreds of feature engineering options with just a few lines of code. There are many situations were this is a very useful process - but as we'll see below, automatic feature engineering without care can lead to less interpretable features.

Run the cell below to generate a significantly larger dataset of engineered features. This process may be slow - you can load in the pretrained results of the engineering process by setting `load_pretrained = True`

In [ ]:
import featuretools as ft

load_pretrained = False

def make_entity_set(X):
    es = ft.EntitySet(id="houses")
    
    X_with_ind = X.copy()
    X_with_ind['index'] = X_with_ind.index
    
    es = es.add_dataframe(
      dataframe_name="houses",
      dataframe=X_with_ind,
      index="index",
    )
    
    return es

def feature_engineer(es, num_to_engineer=15, primitives=None):
    # To avoid large computation times, we 
    ignore_features = list(feature_df["name"][num_to_engineer:])
    
    if primitives is None:
        # To avoid large computation times, we'll just work with a truncated set of options
        primitives = ['multiply_numeric', "greater_than", "divide_by_feature", "cosine", "tangent"]
  
    feature_matrix, feature_defs = ft.dfs(
        entityset=es,
        target_dataframe_name="houses",
        trans_primitives=primitives,
        ignore_columns={"houses": ignore_features},
        max_depth=1,
        max_features=1000
    )
    
    return feature_matrix, feature_defs

if load_pretrained:
    X_train_ft = pd.read_csv("X_train_ft.csv")
    X_test_ft = pd.read_csv("X_test_ft.csv")

else:
    print("Starting deep feature synthesis on training data")
    X_train_ft, feature_defs = feature_engineer(make_entity_set(X_train))

    print("Starting feature engineering on testing data")
    X_test_ft = ft.calculate_feature_matrix(feature_defs, make_entity_set(X_test))

X_train_ft_model, X_test_ft_model = one_hot_encode(X_train_ft, X_test_ft)

model_ft = fit_model(X_train_ft_model, y_train)
print("R2 Score (Featuretools):", model_ft.score(X_test_ft_model, y_test))

Once again, try generating explanations on the generated data, and see if you can guess at the flaw in the data. 

In [ ]:
explain(model_ft, X_train_ft_model)
explain_feature(model_ft, X_train_ft_model, "LotShape * OverallQual")

### Basic Feature Transformations

In many datasets, complex feature engineering does not offer a lot (or any!) boost in model performance, while lowering the interpretability of features. In this dataset, we happened to be given data already turned into useful features. Again, we will train a model on a set of features. Then, try generating some explanations and see if you can guess at the data flaw.

In [ ]:
model_orig = fit_model(X_train_std, y_train)
print("R2 Score (Original):", model_orig.score(X_test_std, y_test))

In [ ]:
# Generate an explanation of the PCA model, and see if you can identify the data flaw
explain(model_orig, X_train_std)
explain_feature(model_orig, X_train_std, "OverallQual")

### Boosting Feature Interpretability

You may have already guessed at one or both flaws in the data based on the explanations generated above, but we can make explanations that are even more immedietely readable by using a library that factors in feature interpretability like [Pyreal](https://dtail.gitbook.io/pyreal/). The example below shows the setup for a Global Feature Importance explanation, which provides one overall importance value per feature to show you which features are most used by the model generally. 

The explanation you generate in the cell below should help you identify one of the data flaws.

In [ ]:
from pyreal.explainers import GlobalFeatureImportance, LocalFeatureContribution
from pyreal.utils.visualize import plot_top_contributors, swarm_plot

feature_descriptions = dict(zip(feature_df.name, feature_df.description))

categorical_features = X_train.select_dtypes(exclude=[np.number]).columns
ohe = OneHotEncoder(columns=categorical_features)
scaler = DataFrameWrapper(StandardScaler())

ohe.fit(pd.concat((X_train, X_test), axis=0))
scaler.fit(ohe.transform(X_train))

global_explainer = GlobalFeatureImportance(model_orig, x_train_orig=X_train, 
                                           e_algorithm="shap", transformers=[ohe, scaler], 
                                           feature_descriptions=feature_descriptions, 
                                           fit_on_init=True)

explanation = global_explainer.produce()
plot_top_contributors(explanation, n=10)

You can now look at multiple individual explanations individually using `plot_top_contributors`, or you may find it more useful to generate a swarm plot like the ones above using Pyreals [swarm_plot](https://sibyl-ml.dev/pyreal/api_reference/api/pyreal.utils.visualize.swarm_plot.html#pyreal.utils.visualize.swarm_plot) function.

In [ ]:
local_explainer = LocalFeatureContribution(model_orig, x_train_orig=X_train, 
                                           e_algorithm="shap", transformers=[ohe, scaler], 
                                           feature_descriptions=feature_descriptions, 
                                           fit_on_init=True)

local_explanation, x_interpret = local_explainer.produce(X_train[0:1000])

You can now look at multiple individual explanations individually using `plot_top_contributors`, or you may find it more useful to generate a swarm plot like the ones above using Pyreals (swarm_plot)[] function.

In [ ]:
swarm_plot(local_explanation, x_interpret, type="strip")

### Results

As you may have determined from the explanations above, the data has two flaws.

<details>
<summary><font color='blue'> Click to reveal solution </font></summary>The first is an example of **data leakage**, by including a feature for the bid price on houses sold in the past. We won't have access to the information for new houses on the market, so a model that relies heavily on this information will not be useful for setting the original house price.

The second was a mistake in the engineering of features, resulting in higher quality of housing materials leading to lower model predictions. If this mistake does not exist in new data coming in, we can expect model performance to decrease. 
</details>

### Bonus activity

If you'd like to get more experience with generating interpretable features, try finding a dataset on a topic you know a lot about. Generate features based on information you know is important, and then compare the explanations and performance to features generated automatically using a tool like the ones above.